# A4 · Pyogrio & Fiona — reading and writing vector files

**What they are:** two Python doors to **GDAL/OGR**, the C++ library that can read and write almost every GIS file format
(GeoPackage, Shapefile, GeoJSON, FlatGeobuf, KML…). PostGIS also uses GDAL, for rasters.

- **Pyogrio**: fast, reads a whole layer straight into a GeoDataFrame. It is GeoPandas' default engine since version 1.0.
- **Fiona**: older, reads **one feature at a time** (like reading a book page by page). Useful for streaming huge files or fine control.

**Daily picture:** GDAL is a universal plug adapter for travel. Pyogrio and Fiona are two ways of plugging your laptop into it.

**Where this fits in your plan:** you already used `gpd.read_file()` — that was Pyogrio working in the background.
Here you learn to read **only what you need** (columns, rows, area) and to choose the right file format.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

In [ ]:
import pyogrio
import fiona
import geopandas as gpd
print("pyogrio", pyogrio.__version__, "| fiona", fiona.__version__, "| GDAL", pyogrio.__gdal_version_string__)

---
## Level 1 — Basics: look inside a file before opening it

**Goal of this level:** list layers and read a file's description (metadata) without loading all the data.

**Daily-life picture:** Reading the table of contents and the back cover before buying a book.

### Exercise 1.1 — What layers are in this file?

**Step 1 · Purpose.** What is the **purpose** of `pyogrio.list_layers(path)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Just pass the path `GPKG`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
pyogrio.____(GPKG)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Lists every layer (name + geometry type) inside a file.

**Real life:** A colleague sends a 2 GB GeoPackage. You first check what is inside before loading anything.

```python
pyogrio.list_layers(GPKG)
```

</details>

### Exercise 1.2 — Describe one layer

**Step 1 · Purpose.** What is the **purpose** of `pyogrio.read_info(path, layer=...)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

The result is a dictionary. Read keys like `info["crs"]`, `info["features"]`, `info["fields"]`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
info = pyogrio.____(GPKG, layer="houses")
print(info["crs"], info["features"], info["fields"], info["geometry_type"])

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Returns metadata of one layer: CRS, number of features, fields and their types, geometry type, bounds.

**Real life:** Check the CRS and the number of rows of the houses layer before a heavy analysis.

```python
info = pyogrio.read_info(GPKG, layer="houses")
print(info["crs"], info["features"], info["fields"], info["geometry_type"])
```

</details>

### Exercise 1.3 — Read a layer into a GeoDataFrame

**Step 1 · Purpose.** What is the **purpose** of `pyogrio.read_dataframe(path, layer=...)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Same arguments as `gpd.read_file`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
roads = pyogrio.____(GPKG, layer="roads")
roads.head()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Reads a layer into a GeoDataFrame (what `gpd.read_file` does by default).

**Real life:** The standard way to start working with a layer.

```python
roads = pyogrio.read_dataframe(GPKG, layer="roads")
roads.head()
```

</details>

---
## Level 2 — Core tools: read only what you need

**Goal of this level:** filter columns, rows and area while reading, so the computer never loads the rest.

**Daily-life picture:** Ordering only the dishes you want instead of the whole menu.

### Exercise 2.1 — Only some columns

**Step 1 · Purpose.** What is the **purpose** of `read_dataframe(..., columns=[...])`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`columns=["name", "road_type"]`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
small = pyogrio.read_dataframe(GPKG, layer="roads", columns=[____, ____])
small.columns.tolist()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Loads only the listed attribute columns (plus geometry).

**Real life:** A national building layer has 80 columns; you need 2. Loading fewer columns saves memory and time.

```python
small = pyogrio.read_dataframe(GPKG, layer="roads", columns=["name", "road_type"])
small.columns.tolist()
```

</details>

### Exercise 2.2 — Only some rows (SQL filter)

**Step 1 · Purpose.** What is the **purpose** of `read_dataframe(..., where="...")`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`where="road_type = 'primary'"` — text values in single quotes inside the double-quoted string.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
primary = pyogrio.read_dataframe(GPKG, layer="roads", where="____")
primary[["name", "road_type"]]

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Filters rows while reading with an SQL-like condition, so only matching features are loaded.

**Real life:** Load only primary roads, or only buildings built after 2000, from a huge file.

```python
primary = pyogrio.read_dataframe(GPKG, layer="roads", where="road_type = 'primary'")
primary[["name", "road_type"]]
```

</details>

### Exercise 2.3 — Only one area (spatial filter)

**Step 1 · Purpose.** What is the **purpose** of `read_dataframe(..., bbox=(minx, miny, maxx, maxy))`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Oldtown is the centre square: (392000, 5820000, 394000, 5822000). The box must be in the layer's CRS.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
old_houses = pyogrio.read_dataframe(GPKG, layer="houses", bbox=(____, ____, ____, ____))
print(len(old_houses), "of", info["features"], "houses loaded")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Reads only features that intersect a box, using the file's spatial index when it has one.

**Real life:** Load only the buildings of your study area from a country-wide file.

```python
old_houses = pyogrio.read_dataframe(GPKG, layer="houses", bbox=(392_000, 5_820_000, 394_000, 5_822_000))
print(len(old_houses), "of", info["features"], "houses loaded")
```

Use `mask=polygon` instead of `bbox` for an exact shape.

</details>

### Exercise 2.4 — Attributes only, no geometry

**Step 1 · Purpose.** What is the **purpose** of `read_dataframe(..., read_geometry=False)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`read_geometry=False`. Then `value_counts()` on `severity`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
table = pyogrio.read_dataframe(GPKG, layer="accidents", read_geometry=____)
print(type(table).__name__)
table["severity"].value_counts()

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Loads the attribute table only, as a normal pandas DataFrame (faster, less memory).

**Real life:** You only need a count per category; the shapes are not needed.

```python
table = pyogrio.read_dataframe(GPKG, layer="accidents", read_geometry=False)
print(type(table).__name__)
table["severity"].value_counts()
```

</details>

---
## Level 3 — Combining: write, convert formats, stream with Fiona

**Goal of this level:** write files in several formats, meet their limits, and read feature by feature.

**Daily-life picture:** Saving a document as .docx, .pdf or .txt: each format keeps some things and loses others.

In [ ]:
OUT = DATA_DIR / "a4_out"
OUT.mkdir(exist_ok=True)
schools = pyogrio.read_dataframe(GPKG, layer="schools")

### Exercise 3.1 — Write a GeoJSON for the web

**Step 1 · Purpose.** What is the **purpose** of `pyogrio.write_dataframe(gdf, path)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

GeoJSON should be in EPSG:4326. So `schools.to_crs(4326)` first, then write to `OUT / "schools.geojson"`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
pyogrio.____(schools.to_crs(____), OUT / "schools.geojson")
print((OUT / "schools.geojson").read_text()[:300])

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Writes a GeoDataFrame to a file; the format is chosen from the file extension (or `driver=`).

**Real life:** Sending schools to a web developer who builds an online map (web maps expect GeoJSON in lon/lat).

```python
pyogrio.write_dataframe(schools.to_crs(4326), OUT / "schools.geojson")
print((OUT / "schools.geojson").read_text()[:300])
```

</details>

### Exercise 3.2 — The Shapefile trap: 10-letter column names

**Step 1 · Purpose.** What is the **purpose** of `write_dataframe(..., 'x.shp')`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Add a long column name, write to `.shp`, read it back and print the columns.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
s = schools.copy()
s["distance_to_river_m"] = 123.0
pyogrio.write_dataframe(s, OUT / "schools.____")
print(pyogrio.read_dataframe(OUT / "schools.shp").columns.tolist())

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Writing a Shapefile works, but the format cuts column names to 10 characters (and has other old limits).

**Real life:** Your column `distance_to_river_m` comes back as `distance_t` — and a colleague cannot tell what it means.

```python
s = schools.copy()
s["distance_to_river_m"] = 123.0
pyogrio.write_dataframe(s, OUT / "schools.shp")
print(pyogrio.read_dataframe(OUT / "schools.shp").columns.tolist())
```

Other Shapefile limits: 2 GB max, one geometry type per file, 4+ files per layer. **Recommendation: use GeoPackage** unless someone really requires Shapefile.

</details>

### Exercise 3.3 — Add rows to an existing layer

**Step 1 · Purpose.** What is the **purpose** of `write_dataframe(..., append=True)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Write the first 3 schools, then append the other 3 with `append=True`, and count the features.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
p = OUT / "log.gpkg"
pyogrio.write_dataframe(schools.iloc[:3], p, layer="schools")
pyogrio.write_dataframe(schools.iloc[3:], p, layer="schools", append=____)
print(pyogrio.read_info(p, layer="schools")["features"])

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Adds new features to the end of an existing layer instead of overwriting it.

**Real life:** Every week new accident reports arrive; you add them to the same layer.

```python
p = OUT / "log.gpkg"
pyogrio.write_dataframe(schools.iloc[:3], p, layer="schools")
pyogrio.write_dataframe(schools.iloc[3:], p, layer="schools", append=True)
print(pyogrio.read_info(p, layer="schools")["features"])
```

</details>

### Exercise 3.4 — Fiona: read one feature at a time

**Step 1 · Purpose.** What is the **purpose** of `with fiona.open(path, layer=...) as src: for feat in src`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Inside the loop: `feat.properties["school"]`, `feat.geometry.coordinates`. `src.schema` and `src.crs` describe the layer.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with fiona.____(GPKG, layer="schools") as src:
    print(src.crs, src.schema)
    for feat in src:
        print(feat.properties["____"], feat.geometry.coordinates)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Opens a layer as a stream; each loop step gives one feature (properties + geometry) without loading the whole file.

**Real life:** A 50 GB file that does not fit in memory: stream through it and keep only what you need.

```python
with fiona.open(GPKG, layer="schools") as src:
    print(src.crs, src.schema)
    for feat in src:
        print(feat.properties["school"], feat.geometry.coordinates)
```

</details>

---
## Level 4 — Professional: data engineering

**Goal of this level:** handle big files, check data quality automatically, and choose formats with reasons.

**Daily-life picture:** A restaurant kitchen: checking every delivery before cooking, and storing food in the right containers.

### Exercise 4.1 — A file too big for memory

**Scenario.** Pretend `houses` has 100 million rows. Count residents per neighbourhood by reading **500 features at a time**.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Loop with `skip_features=start, max_features=500` until the chunk is empty. Add up `groupby("nb_id")["residents"].sum()` from each chunk.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
import pandas as pd
total = pd.Series(dtype=float)
start = 0
while True:
    chunk = pyogrio.read_dataframe(GPKG, layer="houses", read_geometry=False, skip_features=____, max_features=____)
    if len(chunk) == 0:
        break
    total = total.add(chunk.groupby("nb_id")["residents"].sum(), fill_value=0)
    start += 500
total

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Data engineering (chunked processing)

Split, process, combine. Works because a sum can be built from partial sums. (For averages, keep sums **and** counts.)

```python
import pandas as pd
total = pd.Series(dtype=float)
start = 0
while True:
    chunk = pyogrio.read_dataframe(GPKG, layer="houses", read_geometry=False, skip_features=start, max_features=500)
    if len(chunk) == 0:
        break
    total = total.add(chunk.groupby("nb_id")["residents"].sum(), fill_value=0)
    start += 500
total
```

</details>

### Exercise 4.2 — Automatic quality check for any file

**Scenario.** Write a function `check(path, layer)` that reports: CRS missing?, number of features, empty geometries, invalid geometries, duplicate geometries. Run it on every Riverton layer.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Use `read_info` for CRS and count, then load the data and use `.is_empty`, `.is_valid`, `.geometry.duplicated()` (compare WKB). Return a dict; build a DataFrame from all dicts.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
def check(path, layer):
    info = pyogrio.read_info(path, layer=layer)
    g = pyogrio.read_dataframe(path, layer=layer)
    return {"layer": layer, "crs": info["____"], "features": info["features"],
            "empty": int(g.is_empty.sum()), "invalid": int((~g.is_valid).sum()),
            "duplicates": int(g.geometry.to_wkb().duplicated().sum())}
pd.DataFrame([check(GPKG, name) for name, _ in pyogrio.list_layers(GPKG)])

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Data quality (descriptive)

A small, reusable 'health check' run before every analysis catches most data problems early (missing CRS, broken shapes, double entries).

```python
def check(path, layer):
    info = pyogrio.read_info(path, layer=layer)
    g = pyogrio.read_dataframe(path, layer=layer)
    return {"layer": layer, "crs": info["crs"], "features": info["features"],
            "empty": int(g.is_empty.sum()), "invalid": int((~g.is_valid).sum()),
            "duplicates": int(g.geometry.to_wkb().duplicated().sum())}
pd.DataFrame([check(GPKG, name) for name, _ in pyogrio.list_layers(GPKG)])
```

</details>

### Exercise 4.3 — Which format for which job?

**Scenario.** Write the houses layer as GeoPackage, GeoJSON, FlatGeobuf and GeoParquet. Compare file sizes and reading times. Then decide which format to use for (a) sharing with QGIS users, (b) a web map, (c) a big analysis pipeline.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

Loop over `{"gpkg": ..., "geojson": ..., "fgb": ...}` with `write_dataframe`; GeoParquet is written with `houses.to_parquet(...)`. Time reads with `time.perf_counter()`; size with `path.stat().st_size`.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
import time
houses = pyogrio.read_dataframe(GPKG, layer="houses")
rows = []
for ext in ["gpkg", "geojson", "fgb", "parquet"]:
    p = OUT / f"houses.{ext}"
    if ext == "parquet":
        houses.to_parquet(p)
    else:
        pyogrio.write_dataframe(houses if ext != "geojson" else houses.to_crs(4326), p)
    t = time.perf_counter()
    _ = gpd.read_parquet(p) if ext == "parquet" else pyogrio.read_dataframe(p)
    rows.append({"format": ext, "kB": p.stat().st_size // 1024, "read_ms": round(1000 * (time.perf_counter() - t), 1)})
pd.DataFrame(rows)

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Decision (format choice)

(a) **GeoPackage** — one file, many layers, opens everywhere (QGIS, ArcGIS).
(b) **GeoJSON** (small data) or FlatGeobuf (bigger, streamable) — the web understands them.
(c) **GeoParquet** — small and very fast for Python/cloud pipelines (needs `pyarrow`).
Shapefile: only when a partner insists.

```python
import time
houses = pyogrio.read_dataframe(GPKG, layer="houses")
rows = []
for ext in ["gpkg", "geojson", "fgb", "parquet"]:
    p = OUT / f"houses.{ext}"
    if ext == "parquet":
        houses.to_parquet(p)
    else:
        pyogrio.write_dataframe(houses if ext != "geojson" else houses.to_crs(4326), p)
    t = time.perf_counter()
    _ = gpd.read_parquet(p) if ext == "parquet" else pyogrio.read_dataframe(p)
    rows.append({"format": ext, "kB": p.stat().st_size // 1024, "read_ms": round(1000 * (time.perf_counter() - t), 1)})
pd.DataFrame(rows)
```

</details>

---
## 🏁 Final test

For every question: **question type → plan in words → code → interpretation.**

### Test 1 — 🛠️ Task question

**A.** From the accidents layer, load **only fatal accidents** and **only the columns** `date` and `hour`, then save them as GeoJSON for a journalist.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
fatal = pyogrio.read_dataframe(GPKG, layer="accidents", where="severity = 'fatal'", columns=["date", "hour"])
pyogrio.write_dataframe(fatal.to_crs(4326), OUT / "fatal_accidents.geojson")
print(len(fatal))
```

</details>

### Test 2 — 🛠️ Task question

**B.** Load only the houses of **Harbour** (the north-east square: x 394 000–396 000, y 5 822 000–5 824 000) and count them. Then do it with `where="nb_id = 9"` and compare.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
a = pyogrio.read_dataframe(GPKG, layer="houses", bbox=(394_000, 5_822_000, 396_000, 5_824_000))
b = pyogrio.read_dataframe(GPKG, layer="houses", where="nb_id = 9")
print(len(a), len(b))
```
Both give the same houses here, because the neighbourhoods are squares. For real, irregular districts use `mask=polygon` or `where` on a code.

</details>

### Test 3 — 🛠️ Task question

**C.** Convert every layer of `riverton.gpkg` into a separate **FlatGeobuf** file in `OUT`.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
for name, _ in pyogrio.list_layers(GPKG):
    pyogrio.write_dataframe(pyogrio.read_dataframe(GPKG, layer=name), OUT / f"{name}.fgb")
print(sorted(p.name for p in OUT.glob("*.fgb")))
```

</details>

### Test 4 — 🗺️ Decision question

**D.** The city's open-data portal offers the same buildings as Shapefile (ZIP), GeoJSON and GeoPackage. Which do you download and why?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

GeoPackage: one file, no 10-character names, keeps the CRS, has a spatial index for fast `bbox` reads. GeoJSON only if it is small. Shapefile last.

</details>

### Test 5 — 🧠 Modelling question

**E · Modelling a data pipeline.** Every month, the police send a CSV of accidents with lon/lat. Design the steps to turn it into a clean, growing GeoPackage layer that the analysis in A3 can use.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

1. Read the CSV (pandas) → `points_from_xy(lon, lat)`, `crs=4326`.
2. Check: missing coordinates, points outside the town box, duplicates (same time + place).
3. `to_crs(32633)`.
4. `write_dataframe(..., append=True)` to `accidents` in the GeoPackage; keep a `source_month` column.
5. Log how many rows were added or rejected.
This is a small **ETL** (Extract, Transform, Load) process — the same idea as loading data into PostGIS later.

</details>

---
## 🏗️ Project — Clean a messy data delivery

**The brief.** A contractor delivered four files about street furniture. Each one has a typical problem: a missing CRS, a broken polygon,
        duplicates and empty geometries, swapped lon/lat, and a wrong CRS label. Inspect with **Pyogrio**, clean with GeoPandas,
        and deliver **one clean GeoPackage** (EPSG:32633) + PostGIS tables. **Database note:** the last task publishes your result to PostGIS with `publish()` (a small helper around `to_postgis`, explained in B4). The database is the one you started in `00_START_HERE`. If it is not running yet, skip that code and open the saved GeoPackage in QGIS instead.

**Your data** (made for this project, in `data/projects/`):

| Layer / file | What it holds |
|---|---|
| `delivery/playgrounds.shp` | 13 playground polygons — no `.prj` file (CRS unknown). The contractor says: 'it is UTM 33N'. |
| `delivery/benches.geojson` | benches in WGS84 — with duplicates and one empty geometry |
| `delivery/trees.csv` | trees with lon/lat — some rows have lon and lat swapped |
| `delivery/bins.gpkg` | litter bins — labelled EPSG:4326, but the numbers look like metres |

**How to work:** for each task write the plan in words, then the code, then one sentence with the result. Model answers are hidden; open them only after you tried. The last part happens in **QGIS**.

In [ ]:
from geotrain.projects import build_projects, PROJ_DIR, publish
build_projects()                      # writes data/projects/ the first time
print(sorted(p.name for p in PROJ_DIR.iterdir()))

import pandas as pd
import geopandas as gpd
DEL = PROJ_DIR / "delivery"
print(sorted(p.name for p in DEL.iterdir()))

### Project task 1

**Inventory.** For every file, show CRS, feature count, geometry type and bounds (`pyogrio.read_info`). Which problems can you already see?

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
rows = []
for f in ["playgrounds.shp", "benches.geojson", "bins.gpkg"]:
    i = pyogrio.read_info(DEL / f)
    rows.append({"file": f, "crs": i["crs"], "features": i["features"], "type": i["geometry_type"], "bounds": tuple(round(b) for b in i["total_bounds"])})
pd.DataFrame(rows)
```
Playgrounds: CRS `None`. Bins: CRS 4326 but bounds like 390 000 → impossible for degrees.

</details>

### Project task 2

**Playgrounds:** set the CRS (EPSG:32633), find and repair invalid polygons.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
import shapely
play = pyogrio.read_dataframe(DEL / "playgrounds.shp").set_crs(32633)
print("invalid:", (~play.is_valid).sum())
play["geometry"] = shapely.make_valid(play.geometry.values)
print("invalid after:", (~play.is_valid).sum())
```

</details>

### Project task 3

**Benches:** drop empty geometries and exact duplicates, then reproject to EPSG:32633.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
ben = pyogrio.read_dataframe(DEL / "benches.geojson")
n0 = len(ben)
ben = ben[~ben.is_empty & ben.geometry.notna()]
ben = ben[~ben.assign(w=ben.geometry.to_wkb()).duplicated(["bench_id", "w"])].to_crs(32633)
print(n0, "->", len(ben))
```

</details>

### Project task 4

**Trees:** detect swapped rows (Riverton's latitude ≈ 52.5, longitude ≈ 13.4), fix them, and build points.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
trees = pd.read_csv(DEL / "trees.csv")
swapped = trees["lon"] > trees["lat"]
print("swapped rows:", swapped.sum())
trees.loc[swapped, ["lon", "lat"]] = trees.loc[swapped, ["lat", "lon"]].values
trees = gpd.GeoDataFrame(trees, geometry=gpd.points_from_xy(trees.lon, trees.lat), crs=4326).to_crs(32633)
print(trees.total_bounds.round())
```

</details>

### Project task 5

**Bins:** the label is wrong, the numbers are right → **override** the CRS (do not transform!). Then write all four clean layers to one GeoPackage, check it, and publish.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
bins = pyogrio.read_dataframe(DEL / "bins.gpkg").set_crs(32633, allow_override=True)
clean = PROJ_DIR / "a4_street_furniture_clean.gpkg"
clean.unlink(missing_ok=True)
for name, g in {"playgrounds": play, "benches": ben, "trees": trees, "bins": bins}.items():
    pyogrio.write_dataframe(g, clean, layer=name)
print(pyogrio.list_layers(clean))
publish({"furniture_playgrounds": play, "furniture_benches": ben, "furniture_trees": trees, "furniture_bins": bins})
```
Key lesson: `set_crs(allow_override=True)` **re-labels**; `to_crs` **transforms**. Using the wrong one moves your data to the wrong continent.

</details>

### 🗺️ Project in QGIS

1. First drag the **raw** `delivery/playgrounds.shp` into QGIS: QGIS asks for / guesses a CRS — see why a missing `.prj` is dangerous.
2. Drag the raw `bins.gpkg`: it lands far outside the map (it believes the metres are degrees). Remove it.
3. Load the clean layers from **geotrain → projects** (`furniture_*`) or from the clean GeoPackage: all four sit in Riverton.
4. Run *Vector → Geometry Tools → Check Validity* on the clean playgrounds: no errors.

<details><summary>🔌 <b>How to connect QGIS to PostGIS</b> (once per computer)</summary>

1. Start the database (Docker: `docker start geotrain-db`).
2. In QGIS, open the **Browser** panel → right-click **PostgreSQL** → **New Connection…**
3. Name `geotrain`, Host `localhost`, Port `5432`, Database `geotrain`.
   Authentication → *Basic*: user `geo`, password `geo` (tick *Store*). Click **Test Connection** → OK.
4. Expand **geotrain → projects** (the schema) and **drag a table onto the map**.
5. To run SQL and see the result on the map: **Database → DB Manager → PostGIS → geotrain → SQL Window**.
   Write the query → **Execute** → tick **Load as new layer**, choose the geometry column and a unique id column → **Load**.
6. To open a GeoPackage instead: drag the `.gpkg` file from your file manager into QGIS.

</details>

### 📦 Deliverables (tick when done)

- [ ] Inventory table with the problem of each file
- [ ] Clean GeoPackage with 4 layers in EPSG:32633
- [ ] Short cleaning report (what you changed and why) for the contractor

### 🪞 Reflection — your long-term plan

Which file formats does your own data come in? Write the one-line Pyogrio command you would use to read only your study area.

✍️ **Your answer:** …